
## Deep Seek Ideas


In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim

from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaForCausalLM, LlamaTokenizer
from torch.utils.data import DataLoader, Dataset
from accelerate import init_empty_weights, load_checkpoint_and_dispatch


import numpy as np

import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel


import re
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer



In [ ]:

vocab_size  = 100     # Small vocab for synthetic data
embed_size  = 128
num_heads   = 4
num_layers  = 2
hidden_dim  = 256
max_seq_len = 32
seq_len     = 16
batch_size  = 32
epochs      = 10
lr          = 1e-3



## Rewards 


In [ ]:

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]


In [ ]:

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]


In [ ]:


def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses] 
    return [0.5 if match else 0.0 for match in matches]


In [ ]:

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses] 
    return [0.5 if match else 0.0 for match in matches]


In [ ]:

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count


In [ ]:

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]


In [ ]:

class RewardModel(nn.Module):
    def __init__(self, embed_size, hidden_dim):
        super(RewardModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(embed_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, embeddings):
        return self.fc(embeddings).squeeze(-1)


In [ ]:


# Rule-based Reward Function for Multi-Step Reasoning

def rule_based_reward(output_text, expected_answer=None, task_type="reasoning"):
    
    reward = 0.0

    # Format Reward: Check for proper reasoning structure
    if "<think>" in output_text and "</think>" in output_text:
        reward += 0.3  # Reward for using the correct format

    # Step-by-Step Reward: Check intermediate steps
    steps = [segment.strip() for segment in output_text.split("<think>") if "</think>" in segment]
    for step in steps:
        if step in expected_answer:  # Check if the step matches the expected reasoning
            reward += 0.2 / len(steps)  # Reward each correct step proportionally

    # Final Answer Reward: Check for correct answer
    if "[answer]" in output_text and "[/answer]" in output_text:
        answer = extract_answer(output_text)
        if answer == extract_answer(expected_answer):
            reward += 0.5

    return reward





## Data


In [ ]:

# Load and prep dataset

SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""


In [ ]:

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()


In [ ]:

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip().replace(",", "").replace("$", "")


In [ ]:


# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            #{'role': 'user', 'content': 'What is the largest single-digit prime number?'},
            #{'role': 'assistant', 'content': XML_COT_FORMAT.format(
            #    reasoning="9 is divisble by 3 and 8 is divisible by 2, but 7 is prime.",
            #    answer="7"
            #)},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore



In [ ]:

dataset = get_gsm8k_questions()


In [ ]:


def extract_answer(output_text):
    
    if "[answer]" in output_text and "[/answer]" in output_text:
        start = output_text.find("[answer]") + len("[answer]")
        end = output_text.find("[/answer]")
        return output_text[start:end].strip()
    return None



In [ ]:

def generate_synthetic_data(batch_size, seq_len, vocab_size):
    
    # Generate random token sequences
    seq_a = torch.randint(0, vocab_size, (batch_size, seq_len))
    seq_b = torch.randint(0, vocab_size, (batch_size, seq_len))
    # Randomly assign preferences (1 means seq_a preferred over seq_b, 0 otherwise)
    preferences = torch.randint(0, 2, (batch_size,))
    return seq_a, seq_b, preferences


In [ ]:

# Dataset for Multi-Step Reasoning

data = [
    {
        "input": "Why is the sky blue?",
        "output": (
            "<think>Step 1: Sunlight contains all colors of light.</think> "
            "<think>Step 2: As sunlight passes through the atmosphere, it interacts with air molecules.</think> "
            "<think>Step 3: Shorter wavelengths, like blue, scatter more than longer wavelengths, like red.</think> "
            "[answer]Rayleigh scattering[/answer]"
        )
    },
    {
        "input": "What is 2+2?",
        "output": (
            "<think>Step 1: Start with the first number: 2.</think> "
            "<think>Step 2: Add the second number: 2.</think> "
            "<think>Step 3: The result of the addition is 4.</think> "
            "[answer]4[/answer]"
        )
    }
]





In [ ]:

prompts = ["Why is the sky blue?", "What is 2+2?"]
expected_answers = [
    (
        "<think>Step 1: Sunlight contains all colors of light.</think> "
        "<think>Step 2: As sunlight passes through the atmosphere, it interacts with air molecules.</think> "
        "<think>Step 3: Shorter wavelengths, like blue, scatter more than longer wavelengths, like red.</think> "
        "[answer]Rayleigh scattering[/answer]"
    ),
    (
        "<think>Step 1: Start with the first number: 2.</think> "
        "<think>Step 2: Add the second number: 2.</think> "
        "<think>Step 3: The result of the addition is 4.</think> "
        "[answer]4[/answer]"
    )
]


In [ ]:

dataset = [
    {"input": "Why is the sky blue?", "output": "<think>...reasoning...</think> [answer]Rayleigh scattering[/answer]"},
    {"input": "What is 2+2?", "output": "<think>...reasoning...</think> [answer]4[/answer]"}
]



In [ ]:

# Create Dataset Class

class ReasoningDataset(Dataset):
    
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[ idx ]


In [ ]:

# Example data
data = [  # Dummy data: sequence input and target
    (torch.randint(0, vocab_size, (4, block_size)), torch.randint(0, vocab_size, (4, block_size)))
]



## Models


In [ ]:

class GPT(nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_layers, hidden_dim, max_seq_len):
        super(GPT, self).__init__()
        self.embedding           = nn.Embedding(vocab_size, embed_size)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_seq_len, embed_size))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(embed_size, num_heads, hidden_dim),
            num_layers
        )
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        return self.fc(x)


In [ ]:

# GPT Architecture (compatible with pre-trained weights)

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_layer, n_head):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.size()
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits


In [ ]:

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [ ]:

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.query = nn.Linear(n_embd, n_embd)
        self.key = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.register_buffer("mask", torch.tril(torch.ones(1024, 1024)))

    def forward(self, x):
        B, T, C = x.size()
        q = self.query(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.key(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)



In [ ]:

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x)



## Load GPT-2 Pre-Trained Weights


In [ ]:


model_name = "gpt2"
tokenizer  = GPT2Tokenizer.from_pretrained(model_name)
gpt2_model = GPT2LMHeadModel.from_pretrained(model_name)



In [ ]:

# Access weights

gpt2_weights = gpt2_model.state_dict()


In [ ]:

# Initialize Custom GPT Model

vocab_size = gpt2_weights["transformer.wte.weight"].shape[0]
block_size = gpt2_model.config.n_ctx
n_embd     = gpt2_model.config.n_embd
n_layer    = gpt2_model.config.n_layer
n_head     = gpt2_model.config.n_head


In [ ]:

model = GPT(vocab_size, block_size, n_embd, n_layer, n_head)

# Map GPT-2 weights to custom GPT model
model.token_embedding.weight.data    = gpt2_weights["transformer.wte.weight"].clone()
model.position_embedding.weight.data = gpt2_weights["transformer.wpe.weight"].clone()



## Instantiate Model


In [ ]:

#model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_name  = "Qwen/Qwen2.5-1.5B-Instruct"

if "Llama" in model_name:
    output_dir = "outputs/Llama-1B-GRPO"
    run_name = "Llama-1B-GRPO-gsm8k"
else:
    output_dir="outputs/Qwen-1.5B-GRPO"
    run_name="Qwen-1.5B-GRPO-gsm8k"


In [ ]:

model_name = 

model_type = "gpt2"

mixed_precision=True



In [ ]:


if model_type == "gpt2":
    
    tokenizer           = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = self.tokenizer.eos_token  # Add padding token for GPT-2
    
    model = AutoModelForCausalLM.from_pretrained(
              model_name, 
              torch_dtype = torch.float16 if mixed_precision else torch.float32
    ).cuda()
    
elif model_type == "llama":
    
    tokenizer = LlamaTokenizer.from_pretrained(model_name)
    with init_empty_weights():
        model = LlamaForCausalLM.from_pretrained(
                    model_name, 
                    torch_dtype=torch.float16 if mixed_precision else torch.float32
        )
        model = load_checkpoint_and_dispatch(
                model, 
                model_name, 
                device_map = "auto", 
                offload_folder = "offload"
            )
else:
    raise ValueError("Unsupported model type. Use 'gpt2' or 'llama'.")

    model.gradient_checkpointing_enable()
        




## RL functions


In [ ]:


def train_dpo(gpt_model, reward_model, optimizer_gpt, optimizer_reward, vocab_size, seq_len, epochs, batch_size):
    
    for epoch in range(epochs):
        seq_a, seq_b, preferences = generate_synthetic_data(batch_size, seq_len, vocab_size)
        logits_a = gpt_model(seq_a)
        logits_b = gpt_model(seq_b)
        reward_a = reward_model(logits_a.mean(dim=1))
        reward_b = reward_model(logits_b.mean(dim=1))

        loss_dpo_gpt = dpo_loss(reward_a, reward_b, preferences)
        optimizer_gpt.zero_grad()
        loss_dpo_gpt.backward()
        optimizer_gpt.step()

        # Recompute logits for the Reward Model update
        logits_a = gpt_model(seq_a).detach()  # Detach to avoid tracking gradients for GPT again
        logits_b = gpt_model(seq_b).detach()

        # Forward pass through the reward model
        reward_a = reward_model(logits_a.mean(dim=1))
        reward_b = reward_model(logits_b.mean(dim=1))

        # Calculate DPO loss and backpropagate for reward model
        loss_dpo_reward = dpo_loss(reward_a, reward_b, preferences)
        optimizer_reward.zero_grad()
        loss_dpo_reward.backward()
        optimizer_reward.step()
        print(f"Epoch {epoch + 1}/{epochs}, Loss (GPT): {loss_dpo_gpt.item()}, Loss (Reward): {loss_dpo_reward.item()}")




In [ ]:



def fine_tune_with_rl( prompts, expected_answers, num_epochs=1, batch_size=2, learning_rate=1e-5, clip_epsilon=0.2):
    optimizer = optim.AdamW(  model.parameters(), lr=learning_rate  )
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]
            batch_answers = expected_answers[i:i + batch_size]
            generated_texts = [self.generate(prompt) for prompt in batch_prompts]
            rewards = torch.tensor([
                self.reward_fn(output, expected) for output, expected in zip(generated_texts, batch_answers)
            ], dtype=torch.float32).to("cuda")
            old_log_probs = []    ## Compute old and new log probabilities
            new_log_probs = []
            for prompt, generated_text in zip(batch_prompts, generated_texts):
                inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
                outputs = self.model(**inputs, labels=inputs.input_ids)
                old_log_probs.append(outputs.logits.mean().detach())
                generated_inputs  = tokenizer(generated_text, return_tensors="pt").to("cuda")
                generated_outputs = model(**generated_inputs, labels=generated_inputs.input_ids)
                new_log_probs.append(generated_outputs.logits.mean())
            old_log_probs = torch.tensor(old_log_probs, dtype=torch.float32).to("cuda")
            new_log_probs = torch.tensor(new_log_probs, dtype=torch.float32).to("cuda")
            loss = self.compute_grpo_loss(old_log_probs, new_log_probs, rewards, clip_epsilon)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(prompts):.4f}")

       





## SFT


In [ ]:

 
def supervised_fine_tuning(self, dataset, num_epochs=1, batch_size=2, learning_rate=5e-5):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.AdamW(self.model.parameters(), lr=learning_rate)
    self.model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in dataloader:
            inputs = self.tokenizer(batch["input"], return_tensors="pt", padding=True, truncation=True).to("cuda")
            labels = self.tokenizer(batch["output"], return_tensors="pt", padding=True, truncation=True).input_ids.to("cuda")
            labels[labels == self.tokenizer.pad_token_id] = -100
            outputs = self.model(**inputs, labels=labels)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(dataloader):.4f}")

          

In [ ]:

# Fine-Tuning

def fine_tune(model, data, epochs=3, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        for x, y in data:
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1}, Loss: {loss.item()}")




## Losses


In [ ]:

def dpo_loss( reward_a, reward_b, preferences, beta=0.1 ):
    
    logits = (reward_a - reward_b) / beta
  
  
    loss   = -torch.mean( preferences * torch.log_softmax(logits, dim=0))
   
    
    return loss


In [ ]:

    
def compute_grpo_loss(self, old_log_probs, new_log_probs, rewards, clip_epsilon=0.2):
    """
    Compute the GRPO loss for policy optimization.

    Args:
        old_log_probs (torch.Tensor): Log probabilities from the old policy.
        new_log_probs (torch.Tensor): Log probabilities from the new policy.
        rewards (torch.Tensor): Rewards for the generated outputs.
        clip_epsilon (float): Clipping parameter for PPO-like stability.

    Returns:
        torch.Tensor: GRPO loss.
    """
    
    ratios         =  torch.exp(  new_log_probs - old_log_probs  )
    clipped_ratios =  torch.clamp( 
             ratios, 
             1 - clip_epsilon, 
             1 + clip_epsilon 
    )
    
    loss  =  -torch.min(  ratios * rewards, 
                                   clipped_ratios * rewards
    ).mean()
    
    return loss




## Inference


In [ ]:


def generate(input_text, max_length=50):
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(inputs.input_ids, max_length=max_length, pad_token_id=self.tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

  


## Evaluate


In [ ]:


def evaluate(self, prompts, expected_answers):
    self.model.eval()
    correct_answers = 0
    correct_reasoning = 0

    for prompt, expected in zip(prompts, expected_answers):
        output = self.generate(prompt)
        if extract_answer(output) == extract_answer(expected):
            correct_answers += 1

        steps = [segment.strip() for segment in output.split("<think>") if "</think>" in segment]
        expected_steps = [segment.strip() for segment in expected.split("<think>") if "</think>" in segment]
        correct_reasoning += sum(1 for step in steps if step in expected_steps)

    total_prompts = len(prompts)
    reasoning_accuracy = correct_reasoning / total_prompts
    answer_accuracy = correct_answers / total_prompts

    return {
        "answer_accuracy": answer_accuracy,
        "reasoning_accuracy": reasoning_accuracy,
    }




## Main_loop


In [ ]:

model_name = "meta-llama/Llama-2-7b-hf"

model_type = "llama"  # Change to "gpt2" for GPT models

## model_name = "distilgpt2"


In [ ]:

gpt_model        = GPT(vocab_size, embed_size, num_heads, num_layers, hidden_dim, max_seq_len)


In [ ]:

optimizer_gpt    = optim.Adam(gpt_model.parameters(), lr=lr)
optimizer_reward = optim.Adam(reward_model.parameters(), lr=lr)


In [ ]:


train_dpo(gpt_model, reward_model, optimizer_gpt, optimizer_reward, vocab_size, seq_len, epochs, batch_size)



In [ ]:

dataset = ReasoningDataset(data)


In [ ]:

    
prompts          = [   item["input"]  for item in dataset   ]

expected_answers = [   item["output"] for item in dataset   ]


In [ ]:


model = DeepSeekR1(
    model_name = model_name, 
    model_type = model_type
)


In [ ]:

# Assign reward function for RL

model.reward_fn = rule_based_reward


In [ ]:

## reward_model     = RewardModel(embed_size, hidden_dim)
## vocab_size
reward_model     = RewardModel(vocab_size, hidden_dim)



In [ ]:

print("Starting supervised fine-tuning...")

model.supervised_fine_tuning(
    dataset, 
    num_epochs=1, 
    batch_size=1
)


In [ ]:

print("Starting reinforcement learning...")

model.fine_tune_with_rl(
    prompts, 
    expected_answers, 
    num_epochs=1, 
    batch_size=1
)



In [ ]:

print("Evaluating...")

metrics = model.evaluate(prompts, expected_answers)

## metrics = model.evaluate([x["input"] for x in data], [x["output"] for x in data])

print("Metrics:", metrics)


In [ ]:


# Fine-Tune
fine_tune(model, data)



## TRL


In [ ]:

training_args = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    learning_rate=5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type='cosine',
    logging_steps=1,
    bf16=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=16,
    max_prompt_length=256,
    max_completion_length=786,
    num_train_epochs=1,
    save_steps=100,
    max_grad_norm=0.1,
    report_to="wandb",
    log_on_each_node=False,
)


In [ ]:

peft_config = LoraConfig(
    r=16,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05,
)


In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map=None
).to("cuda")
        
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


In [ ]:

# use peft at your own risk; not working for me with multi-GPU training
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func],
    args=training_args,
    train_dataset=dataset,
    #peft_config=peft_config
)



trainer.train()

